# 28 · Checks E and E2: which page examples to keep

Rules fixed in `PLAN.md` (Check E, Check E2) before any scoring. Each example is scored with all three demo methods
on the 90 m grid, with the demo's own plans (cached in `outputs/llm/demo/`), against notebook 22's answer key: the
plan run on the official maps (CDL, NWI, USPVDB) and NASA POWER. Shown places come from `Engine.answer_masks` and
must equal the places the page reports.

- **Pass:** at least one method shows places with precision ≥ 0.6 and F1 ≥ 0.5 (D3a's bar); an empty answer is not
  a hit.
- **Check E (A4's ten examples):** keep those that pass; of the same question wording, at most one.
- **Check E2 (15 North Carolina candidates):** at most one new example per type; the blog's persona wording if it
  passes, otherwise the highest best-method F1. Added after Check E's.

In [1]:
import json
import os

import numpy as np
import pandas as pd
import torch

from askmap.aether_us import TextEncoder
from askmap.catalogue import catalogue_text
from askmap.config import PROJECT_ROOT, data_root, load_config, load_env
from askmap.demo.engine import Engine
from askmap.demo.serve import load_tiles, make_contexts, make_planner
from askmap.llm import make_client, system_prompt, translate
from askmap.plan import execute
from askmap.score import pooled_counts

load_env()
cfg = load_config(PROJECT_ROOT / "configs" / "feasibility.yaml")
AU = load_config(PROJECT_ROOT / "configs" / "aether_us.yaml")
CAP = load_config(PROJECT_ROOT / "configs" / "captions.yaml")
DEMO = data_root() / "demo"
RES = PROJECT_ROOT / "outputs" / "results"
meta = json.loads((DEMO / "meta.json").read_text())
llm = cfg["llm"]
client = make_client(llm["base_url"])
system = system_prompt((PROJECT_ROOT / "configs" / "language.md").read_text(), catalogue_text(cfg))
planner = make_planner(
    cfg,
    lambda q, path: translate(client, llm["models"]["main"], system, q, path, llm),
    PROJECT_ROOT / "outputs" / "llm" / "demo",
)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
enc = TextEncoder(meta["encoder"], meta["encoder_kind"], device, os.environ["HF_HOME"])
engine = Engine(
    load_tiles(DEMO, meta),
    cfg["classes"]["leaves"],
    enc,
    planner,
    make_contexts(cfg, meta, DEMO),
    meta["best"],
    CAP,
    top_share=meta["top_share"],
    not_ready=meta["not_ready"],
)
METHODS = {"M1": "aligned", "M2": "aligned_land_power_climate", "M3": "best"}
print("demo land method:", meta["land_method"])

/Volumes/disco2/github/climate-embeddings-crop-integrity/ask-the-map/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1029 15:44:01.348000 24548 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

Loading weights:  77%|███████▋  | 454/590 [00:00<00:00, 4522.31it/s]

Loading weights: 100%|██████████| 590/590 [00:00<00:00, 4743.11it/s]

demo land method: context_more


In [2]:
def score(ex):
    q, area = ex["question"], ex["area"]
    tiles = engine.areas[area]
    row = {"question": q, "area": area, "type": ex.get("type"), "persona": ex.get("persona")}
    for name, method in METHODS.items():
        r = engine.search(q, area, method)
        pred = engine.answer_masks(r["plan"], area, r["used"])
        truth = execute(r["plan"], {t.name: t.leaf.astype(np.int16) for t in tiles}, engine.contexts[area])
        c = pooled_counts(pred, truth, {t.name: t.valid for t in tiles})
        if c["predicted"] != r["facts"]["shown_pixels"]:
            raise AssertionError(f"{name} {q}: answer masks and the shown places differ")
        hit = bool(
            c["predicted"] > 0
            and c["precision"] is not None
            and c["f1"] is not None
            and c["precision"] >= 0.6
            and c["f1"] >= 0.5
        )
        row[name] = {
            "shown": c["predicted"],
            "true": c["true"],
            "precision": c["precision"],
            "f1": c["f1"],
            "hit": hit,
            "plan_words": r["facts"]["plan_words"],
        }
    row["passes"] = any(row[m]["hit"] for m in METHODS)
    row["best_f1"] = max((row[m]["f1"] for m in METHODS if row[m]["hit"]), default=None)
    return row


def table(rows):
    return pd.DataFrame(
        [
            {
                "area": r["area"],
                "question": r["question"][:60],
                **{f"{m} P": r[m]["precision"] for m in METHODS},
                **{f"{m} F1": r[m]["f1"] for m in METHODS},
                "passes": r["passes"],
            }
            for r in rows
        ]
    ).round(2)

## Check E: A4's ten page examples

In [3]:
A4_TEN = [
    {"question": "corn fields where July 2021 had at least 5 days above 35 C", "area": "north_dakota"},
    {"question": "corn fields where July 2021 had at least 5 days above 35 C", "area": "north_carolina"},
    {"question": "cropland where spring 2022 was one of the wettest 10% since 2001", "area": "north_dakota"},
    {
        "question": "grassland where summer 2022 rainfall was in the driest 10% of summers since 2001",
        "area": "north_dakota",
    },
    {"question": "wetlands next to crop fields", "area": "north_dakota"},
    {"question": "hay fields within 200 m of a wetland", "area": "minnesota"},
    {"question": "soybean fields more than 300 m from any wetland or open water", "area": "minnesota"},
    {"question": "grassland near an existing solar farm", "area": "north_carolina"},
    {"question": "wetlands within 100 m of developed land", "area": "north_carolina"},
    {"question": "forest within 500 m of a solar farm", "area": "georgia"},
]
e_rows, seen = [], set()
for ex in A4_TEN:
    row = score(ex)
    row["keep"] = row["passes"] and row["question"] not in seen
    if row["keep"]:
        seen.add(row["question"])
    e_rows.append(row)
(RES / "check_e_examples.json").write_text(json.dumps(e_rows, indent=1, default=str))
table(e_rows).assign(keep=[r["keep"] for r in e_rows])

,area,question,M1 P,M2 P,M3 P,M1 F1,M2 F1,M3 F1,passes,keep
0,north_dakota,corn fields where July 2021 had at least 5 day...,1.00,0.99,0.93,0.27,0.27,0.93,True,True
1,north_carolina,corn fields where July 2021 had at least 5 day...,0.00,NaN,NaN,0.00,1.00,1.00,False,False
2,north_dakota,cropland where spring 2022 was one of the wett...,0.68,0.97,0.95,0.08,0.07,0.95,True,True
3,north_dakota,grassland where summer 2022 rainfall was in th...,0.16,0.82,0.82,0.04,0.07,0.86,True,True
4,north_dakota,wetlands next to crop fields,NaN,NaN,NaN,1.00,1.00,1.00,False,False
5,minnesota,hay fields within 200 m of a wetland,0.59,0.59,0.31,0.39,0.39,0.27,False,False
6,minnesota,soybean fields more than 300 m from any wetlan...,0.95,0.95,0.86,0.32,0.32,0.85,True,True
7,north_carolina,grassland near an existing solar farm,0.28,0.28,0.33,0.31,0.31,0.39,False,False
8,north_carolina,wetlands within 100 m of developed land,0.71,0.71,0.47,0.09,0.09,0.46,False,False
9,georgia,forest within 500 m of a solar farm,0.12,0.12,0.67,0.02,0.02,0.60,True,True


## Check E2: North Carolina candidates across the blog's kinds of question

In [4]:
NC = "north_carolina"
CANDIDATES = [
    ("heat (crop insurance)", True, "Which corn fields had a heatwave during pollination in July 2022?"),
    ("heat (crop insurance)", False, "corn fields where July 2022 had at least 5 days above 35 C"),
    ("heat (crop insurance)", False, "cotton fields where summer 2022 was one of the hottest 10% since 2001"),
    (
        "drought (drought payments)",
        False,
        "soybean fields where summer 2022 rainfall was in the driest 10% since 2001",
    ),
    (
        "drought (drought payments)",
        False,
        "cropland where root-zone soil moisture in July 2010 was in the driest 10% since 2001",
    ),
    (
        "wet spring (prevented planting)",
        False,
        "cropland where spring 2017 was one of the wettest 10% since 2001",
    ),
    ("wetland conservation", True, "Where is cropland right up against wetlands?"),
    ("wetland conservation", False, "cropland within 100 m of a wetland"),
    ("wetland conservation", False, "wetlands more than 500 m from any cropland"),
    ("solar siting", True, "Grassland near existing solar farms, in the sunnier part of the region"),
    ("solar siting", False, "grassland or hay fields within 1 km of an existing solar farm"),
    (
        "building energy",
        False,
        "developed land in the half of the study area with the most cooling degree days in summer 2022",
    ),
    ("land and distance", False, "cotton fields"),
    ("land and distance", False, "forest more than 1 km from any developed land"),
    ("land and distance", False, "soybean fields within 200 m of forest"),
]
e2_rows = [score({"type": t, "persona": p, "question": q, "area": NC}) for t, p, q in CANDIDATES]
chosen = []
for kind in dict.fromkeys(t for t, _, _ in CANDIDATES):
    passing = [r for r in e2_rows if r["type"] == kind and r["passes"]]
    persona = [r for r in passing if r["persona"]]
    if persona or passing:
        chosen.append((persona or sorted(passing, key=lambda r: -r["best_f1"]))[0])
for r in e2_rows:
    r["chosen"] = r in chosen
(RES / "check_e2_candidates.json").write_text(json.dumps(e2_rows, indent=1, default=str))
table(e2_rows).assign(type=[r["type"] for r in e2_rows], chosen=[r["chosen"] for r in e2_rows])

,area,question,M1 P,M2 P,M3 P,M1 F1,M2 F1,M3 F1,passes,type,chosen
0,north_carolina,Which corn fields had a heatwave during pollin...,0.38,0.95,0.88,0.50,0.82,0.85,True,heat (crop insurance),True
1,north_carolina,corn fields where July 2022 had at least 5 day...,0.91,0.91,0.85,0.83,0.83,0.85,True,heat (crop insurance),False
2,north_carolina,cotton fields where summer 2022 was one of the...,0.95,0.95,0.84,0.51,0.50,0.83,True,heat (crop insurance),False
3,north_carolina,soybean fields where summer 2022 rainfall was ...,0.79,0.97,0.80,0.28,0.28,0.79,True,drought (drought payments),False
4,north_carolina,cropland where root-zone soil moisture in July...,0.79,0.96,0.91,0.18,0.19,0.91,True,drought (drought payments),True
5,north_carolina,cropland where spring 2017 was one of the wett...,0.98,0.96,0.92,0.14,0.13,0.92,True,wet spring (prevented planting),True
6,north_carolina,Where is cropland right up against wetlands?,NaN,NaN,NaN,1.00,1.00,1.00,False,wetland conservation,False
7,north_carolina,cropland within 100 m of a wetland,0.63,0.63,0.61,0.05,0.05,0.61,True,wetland conservation,False
8,north_carolina,wetlands more than 500 m from any cropland,0.56,0.56,0.65,0.10,0.10,0.70,True,wetland conservation,True
9,north_carolina,"Grassland near existing solar farms, in the su...",0.22,0.29,0.39,0.21,0.32,0.44,False,solar siting,False


## The page's examples must be exactly what the rules chose

In [5]:
want = [{"question": r["question"], "area": r["area"]} for r in e_rows if r["keep"]] + [
    {"question": r["question"], "area": r["area"]} for r in chosen
]
have = [{"question": x["question"], "area": x["area"]} for x in AU["demo"]["page_examples"]]
if have != want:
    raise AssertionError(f"configs/aether_us.yaml page_examples differ from the rules' choice:\n{want}")
print(f"{len(have)} page examples, as chosen by Checks E and E2")
for x in have:
    print(f"  {x['area']}: {x['question']}")

11 page examples, as chosen by Checks E and E2
  north_dakota: corn fields where July 2021 had at least 5 days above 35 C
  north_dakota: cropland where spring 2022 was one of the wettest 10% since 2001
  north_dakota: grassland where summer 2022 rainfall was in the driest 10% of summers since 2001
  minnesota: soybean fields more than 300 m from any wetland or open water
  georgia: forest within 500 m of a solar farm
  north_carolina: Which corn fields had a heatwave during pollination in July 2022?
  north_carolina: cropland where root-zone soil moisture in July 2010 was in the driest 10% since 2001
  north_carolina: cropland where spring 2017 was one of the wettest 10% since 2001
  north_carolina: wetlands more than 500 m from any cropland
  north_carolina: developed land in the half of the study area with the most cooling degree days in summer 2022
  north_carolina: cotton fields
